# NGÀY 3 & NGÀY 4: TRÍCH XUẤT ĐẶC TRƯNG OFFLINE (RESNET-50 2048-D & DISTILBERT 768-D) CHO 43,982 MẪU


In [6]:
import os, glob, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms, models
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import Dataset, DataLoader

# ── Seed toàn cục (T09) ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Tự động tìm clean_v0.csv & thư mục ảnh (10000.jpg) ───────────
CSV_PATH = glob.glob('/kaggle/input/**/clean_v0*.csv', recursive=True)[0]
df = pd.read_csv(CSV_PATH)
first_img = str(df['image_file'].iloc[0])
IMG_DIR  = os.path.dirname(glob.glob(f'/kaggle/input/**/{first_img}', recursive=True)[0])

print(f"CSV: {CSV_PATH}")
print(f"IMG: {IMG_DIR}")
print(f"Tổng mẫu: {len(df)} | Splits: {df['split'].value_counts().to_dict()}")
print(f"Columns: {list(df.columns)}")


Device: cuda | GPU: Tesla T4
CSV: /kaggle/input/datasets/nttan28/myntra-cleaned-metadata/clean_v0.csv
IMG: /kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/fashion-dataset/images
Tổng mẫu: 43982 | Splits: {'train': 30787, 'test': 6598, 'val': 6597}
Columns: ['id', 'image_file', 'text', 'text_masked', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'label', 'label_idx', 'split']


## T07 & T10: Trích xuất Đặc trưng Ảnh (ResNet-50 $\rightarrow$ 2048-d, tăng tốc bằng FP16 Mixed Precision & Batch 128)


In [7]:
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
image_encoder = nn.Sequential(*list(resnet.children())[:-1])  # bỏ FC -> (B, 2048, 1, 1)
image_encoder = image_encoder.to(DEVICE).eval()

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class ImgExtractDataset(Dataset):
    def __init__(self, df_split):
        self.ids   = df_split['id'].astype(int).tolist()
        self.files = df_split['image_file'].astype(str).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_path = os.path.join(IMG_DIR, self.files[idx])
        try:
            img = Image.open(img_path).convert('RGB')
            broken = 0
        except Exception:
            img = Image.new('RGB', (224, 224))
            broken = 1
        return IMG_TRANSFORM(img), self.ids[idx], broken

def extract_image_features(df_split, split_name):
    ds = ImgExtractDataset(df_split)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

    all_ids, all_feats = [], []
    total_broken = 0

    for i, (imgs, ids, broken) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            feats = image_encoder(imgs).flatten(1).float()   # (B, 2048) float32
        all_feats.append(feats.cpu())
        all_ids.append(ids)
        total_broken += int(broken.sum().item())

        if (i + 1) % 20 == 0 or (i + 1) == len(loader):
            print(f"  Đã xử lý: {min((i+1)*128, len(ds))}/{len(ds)} ảnh", end='\r')

    feats_tensor = torch.cat(all_feats, dim=0)       # (N, 2048)
    ids_tensor   = torch.cat(all_ids, dim=0).long()
    out_path = f'/kaggle/working/image_feats_{split_name}.pt'
    torch.save({'ids': ids_tensor, 'feats': feats_tensor}, out_path)
    print(f"\n✅ [{split_name}] shape={tuple(feats_tensor.shape)} | broken={total_broken} | saved: {out_path}")
    return feats_tensor

for split_name in ['train', 'val', 'test']:
    df_split = df[df['split'] == split_name].reset_index(drop=True)
    print(f"\n--- Extracting Image {split_name} ({len(df_split)} mẫu) ---")
    t0 = time.time()
    extract_image_features(df_split, split_name)
    print(f"    Time: {time.time()-t0:.1f}s")



--- Extracting Image train (30787 mẫu) ---


/tmp/ipykernel_49/1433521035.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


  Đã xử lý: 30787/30787 ảnh
✅ [train] shape=(30787, 2048) | broken=1 | saved: /kaggle/working/image_feats_train.pt
    Time: 436.2s

--- Extracting Image val (6597 mẫu) ---


/tmp/ipykernel_49/1433521035.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


  Đã xử lý: 6597/6597 ảnh
✅ [val] shape=(6597, 2048) | broken=2 | saved: /kaggle/working/image_feats_val.pt
    Time: 97.1s

--- Extracting Image test (6598 mẫu) ---


/tmp/ipykernel_49/1433521035.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


  Đã xử lý: 6598/6598 ảnh
✅ [test] shape=(6598, 2048) | broken=2 | saved: /kaggle/working/image_feats_test.pt
    Time: 97.9s


## T08 & T11: Trích xuất Đặc trưng Văn bản từ cột `text_masked` (DistilBERT `[CLS]` $\rightarrow$ 768-d)


In [8]:
tokenizer    = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
text_encoder = DistilBertModel.from_pretrained('distilbert-base-uncased').to(DEVICE).eval()

BATCH_SIZE = 128

def extract_text_features(df_split, split_name, col='text_masked'):
    # Ưu tiên cột text_masked đã che từ khóa nhãn để chống Data Leakage
    use_col = col if col in df_split.columns else 'text'
    texts = df_split[use_col].fillna(df_split['text']).fillna('').astype(str).tolist()
    ids   = df_split['id'].astype(int).tolist()
    all_feats = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i+BATCH_SIZE]
        enc = tokenizer(batch_texts,
                        max_length=48,
                        padding='max_length',
                        truncation=True,
                        return_tensors='pt').to(DEVICE)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            out = text_encoder(**enc)
            cls = out.last_hidden_state[:, 0, :].float()   # (B, 768) float32
        all_feats.append(cls.cpu())

        if (i // BATCH_SIZE) % 20 == 0 or (i + len(batch_texts)) == len(texts):
            print(f"  Đã xử lý: {i+len(batch_texts)}/{len(texts)} văn bản", end='\r')

    feats_tensor = torch.cat(all_feats, dim=0)      # (N, 768)
    ids_tensor   = torch.tensor(ids, dtype=torch.long)
    out_path = f'/kaggle/working/text_feats_{split_name}.pt'
    torch.save({'ids': ids_tensor, 'feats': feats_tensor}, out_path)
    print(f"\n✅ [{split_name}] ({use_col}) shape={tuple(feats_tensor.shape)} | saved: {out_path}")

for split_name in ['train', 'val', 'test']:
    df_split = df[df['split'] == split_name].reset_index(drop=True)
    print(f"\n--- Extracting Text {split_name} ({len(df_split)} mẫu) ---")
    t0 = time.time()
    extract_text_features(df_split, split_name, col='text_masked')
    print(f"    Time: {time.time()-t0:.1f}s")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_49/3150492276.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):



--- Extracting Text train (30787 mẫu) ---
  Đã xử lý: 30787/30787 văn bản
✅ [train] (text_masked) shape=(30787, 768) | saved: /kaggle/working/text_feats_train.pt
    Time: 11.1s

--- Extracting Text val (6597 mẫu) ---


/tmp/ipykernel_49/3150492276.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


  Đã xử lý: 6597/6597 văn bản
✅ [val] (text_masked) shape=(6597, 768) | saved: /kaggle/working/text_feats_val.pt
    Time: 2.4s

--- Extracting Text test (6598 mẫu) ---


/tmp/ipykernel_49/3150492276.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


  Đã xử lý: 6598/6598 văn bản
✅ [test] (text_masked) shape=(6598, 768) | saved: /kaggle/working/text_feats_test.pt
    Time: 2.4s


## T09 & T12: Kiểm tra Khớp ID 100% giữa `image_feats`, `text_feats` và `split_ids.csv`


In [9]:
for s in ['train', 'val', 'test']:
    df_s  = df[df['split'] == s].reset_index(drop=True)
    img_d = torch.load(f'/kaggle/working/image_feats_{s}.pt')
    txt_d = torch.load(f'/kaggle/working/text_feats_{s}.pt')
    csv_ids = torch.tensor(df_s['id'].astype(int).values, dtype=torch.long)

    match_img_txt = (img_d['ids'] == txt_d['ids']).all().item()
    match_csv     = (img_d['ids'] == csv_ids).all().item()
    print(f"[{s:5s}] N={len(df_s):5d} | img={tuple(img_d['feats'].shape)} | txt={tuple(txt_d['feats'].shape)} | ID Match: {'✅ 100% OK' if (match_img_txt and match_csv) else '❌ MISMATCH'}")


[train] N=30787 | img=(30787, 2048) | txt=(30787, 768) | ID Match: ✅ 100% OK
[val  ] N= 6597 | img=(6597, 2048) | txt=(6597, 768) | ID Match: ✅ 100% OK
[test ] N= 6598 | img=(6598, 2048) | txt=(6598, 768) | ID Match: ✅ 100% OK
